In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import gc
import os
import duckdb
from tqdm import tqdm

In [4]:
df = pd.read_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_1/epd_snomed_202501.csv")


In [5]:
df.head(5)

,YEAR_MONTH,REGIONAL_OFFICE_NAME,REGIONAL_OFFICE_CODE,ICB_NAME,ICB_CODE,PCO_NAME,PCO_CODE,PRACTICE_NAME,PRACTICE_CODE,ADDRESS_1,...,BNF_DESCRIPTION,BNF_CHAPTER_PLUS_CODE,QUANTITY,ITEMS,TOTAL_QUANTITY,ADQUSAGE,NIC,ACTUAL_COST,UNIDENTIFIED,SNOMED_CODE
0,202501,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,THE GREEN WOOD PRACTICE,F82007,89 GUBBINS LANE,...,Xailin 0.2% eye gel,21: Appliances,10.0,3,30.0,0.0,10.56,9.55702,N,2.444311e+16
1,202501,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,THE GREEN WOOD PRACTICE,F82007,89 GUBBINS LANE,...,Xailin 0.2% eye gel,21: Appliances,20.0,1,20.0,0.0,7.04,6.35894,N,2.444311e+16
2,202501,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,THE GREEN WOOD PRACTICE,F82007,89 GUBBINS LANE,...,Xailin Night eye ointment preservative free,21: Appliances,5.0,3,15.0,0.0,8.10,7.33933,N,2.426801e+16
3,202501,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,THE GREEN WOOD PRACTICE,F82007,89 GUBBINS LANE,...,VisuXL Gel eye drops preservative free,21: Appliances,20.0,1,20.0,0.0,14.98,13.51684,N,3.836411e+16
4,202501,LONDON,Y56,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,NHS NORTH EAST LONDON ICB - A3A8R,A3A8R,THE GREEN WOOD PRACTICE,F82007,89 GUBBINS LANE,...,VisuXL Gel eye drops preservative free,21: Appliances,30.0,1,30.0,0.0,22.47,20.26906,N,3.836411e+16


In [9]:
df = df.rename(columns={
    'REGIONAL_OFFICE_NAME': 'REGIONAL_OFFICE',
    'BNF_CHAPTER_PLUS_CODE': 'BNF_CHAPTER_CODE',
    'CHEMICAL_SUBSTANCE_BNF_DESCR': 'DRUG_NAME',
    'BNF_DESCRIPTION': 'DRUG_DESCRIPTION',
    'ADQUSAGE': 'ADQ_USAGE'
})

df.columns

Index(['YEAR_MONTH', 'REGIONAL_OFFICE', 'REGIONAL_OFFICE_CODE', 'ICB_NAME',
       'ICB_CODE', 'PCO_NAME', 'PCO_CODE', 'PRACTICE_NAME', 'PRACTICE_CODE',
       'ADDRESS_1', 'ADDRESS_2', 'ADDRESS_3', 'ADDRESS_4', 'POSTCODE',
       'BNF_CHEMICAL_SUBSTANCE', 'DRUG_NAME', 'BNF_CODE', 'DRUG_DESCRIPTION',
       'BNF_CHAPTER_CODE', 'QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE',
       'NIC', 'ACTUAL_COST', 'UNIDENTIFIED', 'SNOMED_CODE'],
      dtype='str')

In [ ]:
keep_columns = [
    'YEAR_MONTH', 'REGIONAL_OFFICE', 'ICB_NAME', 'ICB_CODE',
    'PRACTICE_NAME', 'PRACTICE_CODE', 'POSTCODE',
    'BNF_CHAPTER_CODE', 'DRUG_NAME',
    'DRUG_DESCRIPTION', 'ITEMS', 'TOTAL_QUANTITY',
    'ADQ_USAGE', 'NIC', 'ACTUAL_COST', 'UNIDENTIFIED'
]

df = df[keep_columns]

In [11]:
df.columns

Index(['YEAR_MONTH', 'REGIONAL_OFFICE', 'ICB_NAME', 'ICB_CODE',
       'PRACTICE_NAME', 'PRACTICE_CODE', 'POSTCODE', 'BNF_CHAPTER_CODE',
       'DRUG_NAME', 'DRUG_DESCRIPTION', 'ITEMS', 'TOTAL_QUANTITY', 'ADQ_USAGE',
       'NIC', 'ACTUAL_COST', 'UNIDENTIFIED'],
      dtype='str')

In [12]:
df['YEAR_MONTH'] = df['YEAR_MONTH'].astype(str)
df['YEAR_MONTH'] = df['YEAR_MONTH'].apply(
    lambda x: x[:4] + '-' + x[4:] if '-' not in x else x
)

In [19]:
print(df[['YEAR_MONTH']].tail())

         YEAR_MONTH
18435629    2025-01
18435630    2025-01
18435631    2025-01
18435632    2025-01
18435633    2025-01


In [14]:
df['ADQ_USAGE'] = df['ADQ_USAGE'].round(2)
df['NIC'] = df['NIC'].round(2)
df['ACTUAL_COST'] = df['ACTUAL_COST'].round(2)

In [20]:
print(df[['ADQ_USAGE', 'NIC', 'ACTUAL_COST']].tail())

          ADQ_USAGE    NIC  ACTUAL_COST
18435629        0.0  22.07        17.67
18435630        0.0  12.30        11.70
18435631        0.0  42.00        39.91
18435632        0.0  10.50         9.99
18435633        0.0   0.66         0.61


In [25]:
df['TOTAL_QUANTITY'] = df['TOTAL_QUANTITY'].astype(int)
print(df[['TOTAL_QUANTITY']].head())

   TOTAL_QUANTITY
0              30
1              20
2              15
3              20
4              30


In [26]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                  ICB_NAME  \
0    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
1    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
2    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
3    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
4    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   

  ICB_CODE            PRACTICE_NAME PRACTICE_CODE POSTCODE BNF_CHAPTER_CODE  \
0      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
1      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
2      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
3      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
4      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   

      DRUG_NAME                             DRUG_DESCRIPTION  ITEMS  \
0  Eye Pr

In [28]:
df['COST_PER_ITEM'] = (df['ACTUAL_COST'] / df['ITEMS']).round(2)
df['NIC_RATIO'] = (df['ACTUAL_COST'] / df['NIC']).round(2)
df['MONTH'] = df['YEAR_MONTH'].str[5:].astype(int)
df['QUARTER'] = df['MONTH'].apply(lambda x: (x - 1) // 3 + 1)

In [29]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                  ICB_NAME  \
0    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
1    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
2    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
3    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
4    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   

  ICB_CODE            PRACTICE_NAME PRACTICE_CODE POSTCODE BNF_CHAPTER_CODE  \
0      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
1      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
2      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
3      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   
4      QMF  THE GREEN WOOD PRACTICE        F82007  RM3 0DR   21: Appliances   

      DRUG_NAME                             DRUG_DESCRIPTION  ITEMS  \
0  Eye Pr

In [30]:
print(df[df['UNIDENTIFIED'] == 'Y'].shape[0])

18303


In [31]:
df[df['UNIDENTIFIED'] == 'Y']

,YEAR_MONTH,REGIONAL_OFFICE,ICB_NAME,ICB_CODE,PRACTICE_NAME,PRACTICE_CODE,POSTCODE,BNF_CHAPTER_CODE,DRUG_NAME,DRUG_DESCRIPTION,ITEMS,TOTAL_QUANTITY,ADQ_USAGE,NIC,ACTUAL_COST,UNIDENTIFIED,COST_PER_ITEM,NIC_RATIO,MONTH,QUARTER
461184,2025-01,EAST OF ENGLAND,NHS HERTFORDSHIRE AND WEST ESSEX INTEGRA,QM7,UNIDENTIFIED DOCTORS,07H998,CM16 6TN,21: Appliances,Emollients,Zeroderm ointment,1,125,0.00,2.57,2.33,Y,2.33,0.91,1,1
462356,2025-01,EAST OF ENGLAND,NHS HERTFORDSHIRE AND WEST ESSEX INTEGRA,QM7,UNIDENTIFIED DOCTORS,07H998,CM16 6TN,01: Gastro-Intestinal System,Lansoprazole,Lansoprazole 30mg gastro-resistant capsules,1,28,42.00,1.12,0.91,Y,0.91,0.81,1,1
462358,2025-01,EAST OF ENGLAND,NHS HERTFORDSHIRE AND WEST ESSEX INTEGRA,QM7,UNIDENTIFIED DOCTORS,06K998,AL8 6JL,01: Gastro-Intestinal System,Omeprazole,Omeprazole 20mg gastro-resistant capsules,1,28,28.00,0.85,0.69,Y,0.69,0.81,1,1
462947,2025-01,EAST OF ENGLAND,NHS HERTFORDSHIRE AND WEST ESSEX INTEGRA,QM7,UNIDENTIFIED DOCTORS,07H998,CM16 6TN,01: Gastro-Intestinal System,Omeprazole,Omeprazole 20mg gastro-resistant capsules,1,28,28.00,0.85,0.69,Y,0.69,0.81,1,1
462948,2025-01,EAST OF ENGLAND,NHS HERTFORDSHIRE AND WEST ESSEX INTEGRA,QM7,UNIDENTIFIED DOCTORS,07H998,CM16 6TN,01: Gastro-Intestinal System,Omeprazole,Omeprazole 20mg gastro-resistant tablets,1,28,28.00,6.00,4.81,Y,4.81,0.80,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18378647,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,UNIDENTIFIED DOCTORS,NKI998,E1 8AA,04: Central Nervous System,Buprenorphine hydrochloride,Espranor 2mg oral lyophilisates,1,11,36.67,9.98,11.10,Y,11.10,1.11,1,1
18378648,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,UNIDENTIFIED DOCTORS,NKI998,E1 8AA,04: Central Nervous System,Buprenorphine hydrochloride,Espranor 2mg oral lyophilisates,1,20,66.67,18.14,19.26,Y,19.26,1.06,1,1
18378649,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,UNIDENTIFIED DOCTORS,NKI998,E1 8AA,04: Central Nervous System,Buprenorphine hydrochloride,Espranor 2mg oral lyophilisates,7,196,653.33,177.80,177.97,Y,25.42,1.00,1,1
18378650,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,UNIDENTIFIED DOCTORS,NKI998,E1 8AA,04: Central Nervous System,Buprenorphine hydrochloride,Espranor 2mg oral lyophilisates,1,22,73.33,19.96,21.20,Y,21.20,1.06,1,1


In [32]:
df = df[df['UNIDENTIFIED'] != 'Y']

In [34]:
print(df['UNIDENTIFIED'].value_counts())

UNIDENTIFIED
N    18417331
Name: count, dtype: int64


In [35]:
print(df.isnull().sum())

YEAR_MONTH             0
REGIONAL_OFFICE        0
ICB_NAME               0
ICB_CODE               0
PRACTICE_NAME          0
PRACTICE_CODE          0
POSTCODE            5286
BNF_CHAPTER_CODE       0
DRUG_NAME              0
DRUG_DESCRIPTION       0
ITEMS                  0
TOTAL_QUANTITY         0
ADQ_USAGE              0
NIC                    0
ACTUAL_COST            0
UNIDENTIFIED           0
COST_PER_ITEM          0
NIC_RATIO            275
MONTH                  0
QUARTER                0
dtype: int64


In [36]:
print(df[df['NIC'] == 0].shape[0])

283


In [37]:
before = len(df)
print(f"Rows before dropping: {before}")


Rows before dropping: 18417331


In [38]:
df = df[df['NIC'] != 0]


In [39]:
after = len(df)
print(f"Rows after dropping: {after}")
print(f"Rows dropped: {before - after}")

Rows after dropping: 18417048
Rows dropped: 283


In [42]:
print(f"NIC = 0: {(df['NIC'] == 0).sum()}")
print(f"NIC missing: {df['NIC'].isnull().sum()}")

NIC = 0: 0
NIC missing: 0


In [47]:
df = df.drop(columns=['POSTCODE'])

In [49]:
print(df.head());

  YEAR_MONTH REGIONAL_OFFICE                                  ICB_NAME  \
0    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
1    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
2    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
3    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   
4    2025-01          LONDON  NHS NORTH EAST LONDON INTEGRATED CARE BO   

  ICB_CODE            PRACTICE_NAME PRACTICE_CODE BNF_CHAPTER_CODE  \
0      QMF  THE GREEN WOOD PRACTICE        F82007   21: Appliances   
1      QMF  THE GREEN WOOD PRACTICE        F82007   21: Appliances   
2      QMF  THE GREEN WOOD PRACTICE        F82007   21: Appliances   
3      QMF  THE GREEN WOOD PRACTICE        F82007   21: Appliances   
4      QMF  THE GREEN WOOD PRACTICE        F82007   21: Appliances   

      DRUG_NAME                             DRUG_DESCRIPTION  ITEMS  \
0  Eye Products                          Xailin 0.2% eye gel   

In [50]:
df.to_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_Main/epd_snomed_202501_clean.csv", index=False)

In [51]:
df = pd.read_csv("D:/UEA/Business Analytics/P1/Code_Dataset/Data_Main/epd_snomed_202501_clean.csv")

In [52]:
df.head(5)

,YEAR_MONTH,REGIONAL_OFFICE,ICB_NAME,ICB_CODE,PRACTICE_NAME,PRACTICE_CODE,BNF_CHAPTER_CODE,DRUG_NAME,DRUG_DESCRIPTION,ITEMS,TOTAL_QUANTITY,ADQ_USAGE,NIC,ACTUAL_COST,UNIDENTIFIED,COST_PER_ITEM,NIC_RATIO,MONTH,QUARTER
0,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,THE GREEN WOOD PRACTICE,F82007,21: Appliances,Eye Products,Xailin 0.2% eye gel,3,30,0.0,10.56,9.56,N,3.19,0.91,1,1
1,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,THE GREEN WOOD PRACTICE,F82007,21: Appliances,Eye Products,Xailin 0.2% eye gel,1,20,0.0,7.04,6.36,N,6.36,0.90,1,1
2,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,THE GREEN WOOD PRACTICE,F82007,21: Appliances,Eye Products,Xailin Night eye ointment preservative free,3,15,0.0,8.10,7.34,N,2.45,0.91,1,1
3,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,THE GREEN WOOD PRACTICE,F82007,21: Appliances,Eye Products,VisuXL Gel eye drops preservative free,1,20,0.0,14.98,13.52,N,13.52,0.90,1,1
4,2025-01,LONDON,NHS NORTH EAST LONDON INTEGRATED CARE BO,QMF,THE GREEN WOOD PRACTICE,F82007,21: Appliances,Eye Products,VisuXL Gel eye drops preservative free,1,30,0.0,22.47,20.27,N,20.27,0.90,1,1
